In [ ]:
#!/usr/bin/env python3

import csv
import json
import os
import time
from datetime import datetime, timezone, timedelta
import pandas as pd
import requests
from tqdm import tqdm

# ============================================================
# CONFIG (UNCHANGED)
# ============================================================
INPUT_CSV = "accidents_final_processed.csv"
OUTPUT_JSONL = "open_meteo_archive_only.jsonl"
ERRORS_CSV = "open_meteo_errors.csv"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
BACKOFF_FACTOR = 2.0
SLEEP_BETWEEN_REQUESTS = 0.0

MONTHLY_CALL_BUDGET = int(os.getenv("OPEN_METEO_CALL_BUDGET", "5000000"))

OPEN_METEO_API_KEY = os.getenv("OPEN_METEO_API_KEY", "").strip()

ARCHIVE_ENDPOINT = os.getenv(
    "OPEN_METEO_ARCHIVE_ENDPOINT",
    "https://archive-api.open-meteo.com/v1/archive",
)

FORECAST_ENDPOINT = os.getenv(
    "OPEN_METEO_FORECAST_ENDPOINT",
    "https://customer-api.open-meteo.com/v1/forecast",
)

TIMEZONE = "UTC"
TIMEFORMAT = "iso8601"
CELL_SELECTION = "land"

ARCHIVE_MODELS = os.getenv("OPEN_METEO_ARCHIVE_MODELS", "best_match")
FORECAST_MODELS = os.getenv("OPEN_METEO_FORECAST_MODELS", "best_match")

ELEVATION_OVERRIDE = os.getenv("OPEN_METEO_ELEVATION_OVERRIDE", "").strip()

BATCH_SIZE = 10

session = requests.Session()
calls_made = 0


# ============================================================
# VARIABLES (UNCHANGED)
# ============================================================

ARCHIVE_HOURLY_VARS = [
    "temperature_2m",
    "relativehumidity_2m",
    "dewpoint_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weathercode",
    "pressure_msl",
    "surface_pressure",
    "cloudcover",
    "cloudcover_low",
    "cloudcover_mid",
    "cloudcover_high",
    "windspeed_10m",
    "winddirection_10m",
    "windgusts_10m",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
    "windspeed_100m",
    "winddirection_100m",
    "soil_temperature_0_to_7cm",
    "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm",
    "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm",
    "soil_moisture_100_to_255cm",
]

ARCHIVE_DAILY_VARS = [
    "weathercode",
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_max",
    "apparent_temperature_min",
    "sunrise",
    "sunset",
    "daylight_duration",
    "sunshine_duration",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "precipitation_hours",
    "windspeed_10m_max",
    "windgusts_10m_max",
    "winddirection_10m_dominant",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

# ============================================================
# OUTPUT COLUMNS (KEPT EXACTLY AS YOU HAD)
# ============================================================
OUTPUT_COLUMNS = [
    "_row_index",
    "accident_time_utc",
    "source_latitude",
    "source_longitude",
    "archive_status",
    "archive_error",
    "archive_request_count",
    "archive_json",
    "forecast_status",
    "forecast_error",
    "forecast_request_count",
    "forecast_json",
]


# ============================================================
# HELPERS
# ============================================================

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def parse_accident_datetime_utc(row):
    raw = str(row.get("accident_time_utc", "")).strip()
    if not raw:
        raise ValueError("missing accident_time_utc")

    return datetime.strptime(
        raw,
        "%Y-%m-%d %H:%M:%S"
    ).replace(tzinfo=timezone.utc)


def get_base_params(lat, lon):
    params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "timezone": TIMEZONE,
        "timeformat": TIMEFORMAT,
        "cell_selection": CELL_SELECTION,
        "models": ARCHIVE_MODELS,
    }

    if ELEVATION_OVERRIDE:
        params["elevation"] = ELEVATION_OVERRIDE

    if OPEN_METEO_API_KEY:
        params["apikey"] = OPEN_METEO_API_KEY

    return params


def request_json(endpoint, params):
    global calls_made

    attempt = 0
    wait = 2.0
    last_url = None

    while attempt < MAX_RETRIES:
        try:
            calls_made += 1
            resp = session.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
            last_url = resp.url

            if resp.status_code == 429:
                time.sleep(60)
                attempt += 1
                continue

            if resp.status_code == 400:
                return False, {"error": resp.text}, resp.url

            resp.raise_for_status()
            return True, resp.json(), resp.url

        except Exception as exc:
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, {"error": "max retries"}, last_url

def merge(a, b):
    if not a:
        a = {}
    for k, v in b.items():
        if k in a and isinstance(a[k], dict) and isinstance(v, dict):
            a[k].update(v)
        else:
            a[k] = v
    return a

def fetch_archive_for_row(row_idx, lat, lon, accident_dt):
    base_params = get_base_params(lat, lon)

    hour = accident_dt.replace(minute=0, second=0, microsecond=0)

    start = (hour - timedelta(days=1)).date().isoformat()
    end = hour.date().isoformat()

    base_params.update({
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(ARCHIVE_HOURLY_VARS),
        "daily": None,
    })

    ok, payload, url = request_json(ARCHIVE_ENDPOINT, base_params)

    archive_meta = {
        "endpoint": ARCHIVE_ENDPOINT,
        "urls": [url] if url else [],
        "requests": 1,
        "errors": [] if ok else ["archive request failed"],
    }

    return ok, archive_meta, payload
# ============================================================
# MAIN LOOP (FORECAST COMPLETELY REMOVED)
# ============================================================

def run():
    df = pd.read_csv(INPUT_CSV, dtype=str)

    completed = set()
    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)["_row_index"])
                except:
                    pass

    success = 0
    errors = 0

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        if idx in completed:
            continue

        try:
            lat = float(row.get("latitude") or row.get("source_latitude"))
            lon = float(row.get("longitude") or row.get("source_longitude"))
            dt = parse_accident_datetime_utc(row)

            ok, meta, payload = fetch_archive_for_row(idx, lat, lon, dt)

            out_row = {
                "_row_index": idx,
                "accident_time_utc": dt.isoformat(),
                "source_latitude": lat,
                "source_longitude": lon,

                "archive_status": "ok" if ok else "partial",
                "archive_error": meta["errors"],
                "archive_request_count": meta["requests"],
                "archive_json": payload,
            }

            with open(OUTPUT_JSONL, "a") as f:
                f.write(json.dumps(out_row) + "\n")

            success += 1

        except Exception as e:
            errors += 1
            with open(ERRORS_CSV, "a") as f:
                f.write(f"{idx},fatal,{str(e)}\n")

        if calls_made >= MONTHLY_CALL_BUDGET:
            break

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f"Done: {success} success, {errors} errors, calls={calls_made}")


if __name__ == "__main__":
    run()

In [ ]:
!pip install azure-storage-blob
#!/usr/bin/env python3

import os
from azure.storage.blob import BlobServiceClient

# =========================
# CONFIG
# =========================
CONNECTION_STRING = ""
CONTAINER_NAME = "london"

LOCAL_FILE = "open_meteo_archive_only.jsonl"
BLOB_PATH = "open_meteo_data/open_meteo_archive_only.jsonl"


def upload_file():
    if not CONNECTION_STRING:
        raise ValueError("Missing AZURE_STORAGE_CONNECTION_STRING")

    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)

    # create container if it doesn't exist
    try:
        container_client.create_container()
    except Exception:
        pass

    blob_client = container_client.get_blob_client(BLOB_PATH)

    with open(LOCAL_FILE, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)

    print(f"Uploaded: {LOCAL_FILE} -> {CONTAINER_NAME}/{BLOB_PATH}")


if __name__ == "__main__":
    upload_file()

In [ ]:
import pandas as pd
import math
import os

# Input CSV file
input_file = "all_59k_accidents_raw.csv"

# Output folder
output_dir = "split_output"
os.makedirs(output_dir, exist_ok=True)

# Read CSV
df = pd.read_csv(input_file)

# Number of splits
num_splits = 10

# Rows per split
rows_per_file = math.ceil(len(df) / num_splits)

# Split and save
for i in range(num_splits):
    start = i * rows_per_file
    end = start + rows_per_file
    chunk = df.iloc[start:end]

    if not chunk.empty:
        output_file = os.path.join(output_dir, f"all_59k_accidents_raw{i+1}.csv")
        chunk.to_csv(output_file, index=False)
        print(f"Saved: {output_file} ({len(chunk)} rows)")

print("Done.")

In [ ]:
import pandas as pd
import os

original = len(pd.read_csv("all_59k_accidents_raw.csv"))
total_split_rows = 0

for file in os.listdir("split_output"):
    if file.endswith(".csv"):
        total_split_rows += len(pd.read_csv(os.path.join("split_output", file)))

print("Original rows:", original)
print("Split rows total:", total_split_rows)
print("Match:", original == total_split_rows)

In [ ]:
# Install dependency
!pip install -q azure-storage-blob

from azure.storage.blob import BlobServiceClient
import os

# =========================
# CONFIGURATION
# =========================
CONNECTION_STRING = "BlobEndpoint=https://YOUR_STORAGE_ACCOUNT.blob.core.windows.net/;SharedAccessSignature=sp=racwlme&st=2026-05-06T22:35:43Z&se=2026-10-10T06:50:43Z&sv=2025-11-05&sr=c&sig=YOUR_AZURE_SAS_SIGNATURE"

CONTAINER_NAME = "london"

# Local file in Colab
LOCAL_FILE_PATH = "open_meteo_archive_and_forecast_full_10.jsonl"

# Azure blob path (virtual folders)
BLOB_NAME = "open_meteo_data/open_meteo_archive_and_forecast_full_10.jsonl"
# Example result:
# container/myfolder/subfolder/your_big_file.zip

# =========================
# VALIDATION
# =========================
if not os.path.exists(LOCAL_FILE_PATH):
    raise FileNotFoundError(f"File not found: {LOCAL_FILE_PATH}")

file_size = os.path.getsize(LOCAL_FILE_PATH) / (1024**2)
print(f"Found file: {LOCAL_FILE_PATH}")
print(f"Size: {file_size:.2f} MB")

# =========================
# CONNECT TO AZURE
# =========================
print("Connecting to Azure Blob Storage...")

blob_service_client = BlobServiceClient.from_connection_string(
    CONNECTION_STRING
)

blob_client = blob_service_client.get_blob_client(
    container=CONTAINER_NAME,
    blob=BLOB_NAME
)

# =========================
# UPLOAD
# =========================
print(f"Uploading to: {CONTAINER_NAME}/{BLOB_NAME}")

with open(LOCAL_FILE_PATH, "rb") as data:
    blob_client.upload_blob(
        data,
        overwrite=True,
        max_concurrency=4  # parallel upload for large files
    )

print("Upload complete.")
print(f"Uploaded blob path: {CONTAINER_NAME}/{BLOB_NAME}")

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================

file1_path = "/content/all_59k_accidents_complete_final_cleaned.csv"
file2_path = "/content/processed_merged.csv"

# FINAL OUTPUT
output_path = "/content/all_59k_accidents_complete_final_cleaned_matched.csv"

# OPTIONAL LOG FILES
matched_output_path = "/content/matched_rows.csv"
unmatched_output_path = "/content/unmatched_rows.csv"

# =========================================================
# LOAD FILES
# =========================================================

print("Loading CSV files...")

df1 = pd.read_csv(file1_path, low_memory=False)
df2 = pd.read_csv(file2_path, low_memory=False)

print(f"File1 rows: {len(df1):,}")
print(f"File2 rows: {len(df2):,}")

# =========================================================
# STANDARDIZE COLUMN NAMES
# file1 uses latitude/longitude
# file2 uses source_latitude/source_longitude
# =========================================================

df1 = df1.rename(columns={
    "latitude": "source_latitude",
    "longitude": "source_longitude"
})

# temporary matching keys only
df1["_match_date"] = pd.to_datetime(
    df1["accidentdate"], format="%d-%b-%y", errors="coerce"
).dt.strftime("%Y-%m-%d")

df1["_match_time"] = pd.to_datetime(
    df1["time"], format="%H:%M", errors="coerce"
).dt.strftime("%H:%M")

df2["_match_date"] = pd.to_datetime(
    df2["accident_time_utc"], utc=True, errors="coerce"
).dt.strftime("%Y-%m-%d")

df2["_match_time"] = pd.to_datetime(
    df2["accident_time_utc"], utc=True, errors="coerce"
).dt.strftime("%H:%M")

# =========================================================
# VALIDATE REQUIRED COLUMNS
# =========================================================

required_cols = ["source_latitude", "source_longitude", "accidentdate", "time"]

for col in required_cols:
    if col not in df1.columns:
        raise ValueError(f"{col} missing from file1")

if "accident_time_utc" not in df2.columns:
    raise ValueError("accident_time_utc missing from file2")

# =========================================================
# FORCE NUMERIC TYPES
# (NO ROUNDING)
# =========================================================

for df in [df1, df2]:
    df["source_latitude"] = pd.to_numeric(
        df["source_latitude"],
        errors="coerce"
    )

    df["source_longitude"] = pd.to_numeric(
        df["source_longitude"],
        errors="coerce"
    )

# =========================================================
# SELECT COLUMNS FROM FILE2 TO APPEND TO FILE1
# Exclude merge keys
# =========================================================

# drop overlapping columns from file1, keep merge keys
merge_keys = ["source_latitude", "source_longitude", "_match_date", "_match_time"]

to_drop = [c for c in df1.columns if c in df2.columns and c not in merge_keys]

print("\nColumns dropped from file1:")
print(to_drop)

df1 = df1.drop(
    columns=to_drop,
    errors="ignore"
)

# =========================================================
# SELECT COLUMNS FROM FILE2 TO APPEND TO FILE1
# Exclude merge keys
# =========================================================

extra_cols = [
    col for col in df2.columns
    if col not in merge_keys
]

print(f"\nColumns being appended from file2 ({len(extra_cols)}):")
print(extra_cols)

# =========================================================
# MERGE
# KEEP ALL ROWS FROM FILE1
# APPEND MATCHING DATA FROM FILE2
# =========================================================

print("\nMerging files...")

df2 = df2.drop_duplicates(
    subset=merge_keys,
    keep="first"
)

merged = df1.merge(
    df2[merge_keys + extra_cols],
    on=merge_keys,
    how="left",
    indicator=True
)

# =========================================================
# MATCH LOGGING
# =========================================================

matched = merged[merged["_merge"] == "both"].copy()
unmatched = merged[merged["_merge"] == "left_only"].copy()

matched_count = len(matched)
unmatched_count = len(unmatched)
total_rows = len(merged)

match_rate = (matched_count / total_rows) * 100

print("\n=================================================")
print("MERGE SUMMARY")
print("=================================================")

print(f"Total rows in output:      {total_rows:,}")
print(f"Matched rows:              {matched_count:,}")
print(f"Unmatched rows:            {unmatched_count:,}")
print(f"Match rate:                {match_rate:.2f}%")

# =========================================================
# SHOW SAMPLE UNMATCHED ROWS
# =========================================================

if unmatched_count > 0:

    print("\nSample unmatched coordinates:")

    print(
        unmatched[
            ["source_latitude", "source_longitude"]
        ]
        .head(20)
        .to_string(index=False)
    )

# =========================================================
# SAVE LOG FILES
# =========================================================

print("\nSaving log files...")

matched.to_csv(matched_output_path, index=False)
unmatched.to_csv(unmatched_output_path, index=False)

print(f"Matched rows log:   {matched_output_path}")
print(f"Unmatched rows log: {unmatched_output_path}")

# =========================================================
# CLEAN OUTPUT
# =========================================================

merged = merged.drop(columns=["_merge", "_match_date", "_match_time"], errors="ignore")

# OPTIONAL:
# Rename columns back to latitude/longitude
# =========================================================

merged = merged.rename(columns={
    "source_latitude": "latitude",
    "source_longitude": "longitude"
})

# =========================================================
# SAVE FINAL FILE
# =========================================================

merged.to_csv(output_path, index=False)

print(f"\nFinal merged file saved:")
print(output_path)

print("\nDONE.")

In [ ]:
import pandas as pd

# =========================================================
# LOAD DATA
# =========================================================

file_path = "/content/all_59k_accidents_complete_final_cleaned_matched.csv"
output_path = "/content/all_59k_accidents_complete_final_2.csv"

df = pd.read_csv(file_path, low_memory=False)

print(f"Initial columns: {len(df.columns)}")

# =========================================================
# COLUMNS TO DROP
# =========================================================

cols_to_drop = [
    'forecast.hourly.visibility (not accident-aligned in this file)',
    'forecast.hourly.visibility',
    'forecast.hourly.precipitation_probability',
    'forecast.hourly.showers',
    'archive.hourly.weathercode / forecast.hourly.weather_code',
    'archive.hourly.snowfall + snow_depth + temperature',
    'archive.hourly.precipitation + rain + snowfall + snow_depth',
    'archive.hourly.precipitation + et0_fao_evapotranspiration',
    'archive.hourly.windgusts_10m + windspeed_10m',
    'archive.hourly.windspeed_100m + windspeed_10m',
    'archive.hourly.windspeed_10m + winddirection_10m',
    'archive.hourly.windspeed_10m + rain',
    'archive.hourly.windgusts_10m + rain',
    'archive.hourly.windgusts_10m + wet flag',
    'archive.daily.sunrise + sunset',
    'archive.daily.sunshine_duration + daylight_duration',
    'cloud/rain derived',
    'archive.hourly.cloudcover_low + relativehumidity_2m',
    'archive.hourly.cloudcover + rain',
    'archive.hourly.apparent_temperature + temperature_2m',
    'archive.hourly.temperature_2m + precipitation',
    'archive.hourly.temperature_2m + windspeed_10m',
    'archive.daily.temperature_2m_max + temperature_2m_min',
    'archive.hourly.temperature_2m + dewpoint_2m',
    'archive.hourly.relativehumidity_2m + precipitation',
    'archive humidity/temp/precip',
    'archive.hourly.surface_pressure + pressure_msl',
    'archive.hourly.soil_temperature_0_to_7cm + precipitation'
]

# =========================================================
# LOG WHAT WILL BE DROPPED
# =========================================================

actually_dropping = [c for c in cols_to_drop if c in df.columns]

print("\nColumns to be dropped:")
print(actually_dropping)

# =========================================================
# DROP
# =========================================================

df = df.drop(columns=actually_dropping, errors="ignore")

# =========================================================
# SAVE CLEAN CSV
# =========================================================

df.to_csv(output_path, index=False)

print(f"\nSaved cleaned file to:")
print(output_path)

print(f"\nRemaining columns: {len(df.columns)}")
print("Done.")

In [ ]:
import pandas as pd
from datetime import datetime, timezone

INPUT_CSV = "all_59k_accidents_complete_final_2_cleaned.csv"

def parse_source_dt(row):
    """Rebuild datetime from original columns"""
    raw_date = str(row["accidentdate"]).strip()
    raw_time = str(row["time"]).strip()

    date_part = datetime.strptime(raw_date, "%d-%b-%y")
    time_part = datetime.strptime(raw_time, "%H:%M")

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,   # same assumption as your script
    )

def parse_output_dt(val):
    """Parse computed UTC column"""
    return datetime.fromisoformat(val.replace("Z", "+00:00"))

def main():
    df = pd.read_csv(INPUT_CSV, dtype=str)

    mismatches = []

    for idx, row in df.iterrows():
        try:
            src_dt = parse_source_dt(row)
            out_dt = parse_output_dt(row["accident_time_utc"])

            if src_dt != out_dt:
                mismatches.append({
                    "row": idx,
                    "accidentdate": row["accidentdate"],
                    "time": row["time"],
                    "expected_utc": src_dt.isoformat(),
                    "actual_utc": row["accident_time_utc"],
                })

        except Exception as e:
            mismatches.append({
                "row": idx,
                "error": str(e),
                "accidentdate": row.get("accidentdate"),
                "time": row.get("time"),
            })

    print(f"\nTotal mismatches: {len(mismatches)}\n")

    # print first 50 for inspection
    for m in mismatches[:50]:
        print(m)

    # optionally save
    pd.DataFrame(mismatches).to_csv("utc_mismatches.csv", index=False)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

# Paths
input_file = "/content/all_59k_accidents_archive_weather_no_index.csv"
output_file = "/content/all_59k_accidents_archive_weather_final.csv"

# Read CSV
df = pd.read_csv(input_file)

# Drop row_index column
df = df.drop(columns=["accident_time_utc"])

# Save CSV
df.to_csv(output_file, index=False)

print(f"Saved cleaned file to: {output_file}")

In [ ]:
import pandas as pd

input_file = "/content/all_59k_accidents_archive_weather_enriched_final.csv"
output_file = "/content/all_59k_accidents_archive_weather_enriched_final_no_nan.csv"

df = pd.read_csv(input_file)

# Find columns where all values are NaN
removed_cols = df.columns[df.isna().all()].tolist()

# Log removed columns
print("Columns removed (all values NaN):")
for col in removed_cols:
    print(f"- {col}")

print(f"\nTotal removed: {len(removed_cols)}")

# Remove them
df = df.drop(columns=removed_cols)

# Save cleaned CSV
df.to_csv(output_file, index=False)

print(f"\nSaved cleaned file to: {output_file}")
print(f"Remaining columns: {len(df.columns)}")

In [ ]:
# Merge all CSV files in a folder into one CSV in Google Colab

import pandas as pd
import glob
import os

# Path to your folder containing CSV files
folder_path = "/content/csv_folder"  # change this

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and combine
df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

# Save merged file
output_file = "/content/processed_merged.csv"
merged_df.to_csv(output_file, index=False)

print(f"Merged {len(csv_files)} files into: {output_file}")

In [ ]:
#!/usr/bin/env python3

import json
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

INPUT_JSONL = "open_meteo_archive_missing.jsonl"
OUTPUT_CSV = "open_meteo_archive_missing.csv"


# ============================================================
# HELPERS
# ============================================================

def first_or_none(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value


# ============================================================
# MAIN
# ============================================================

rows = []

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        obj = json.loads(line)

        row = {}

        # ====================================================
        # TOP-LEVEL FIELDS
        # ====================================================

        for k, v in obj.items():
            if k == "forecast":
                continue

            row[k] = v

        # ====================================================
        # FORECAST SECTION
        # ====================================================

        forecast = obj.get("forecast", {})

        for k, v in forecast.items():

            # Skip nested hourly for separate handling
            if k == "hourly":
                continue

            # Skip hourly_units metadata
            if k == "hourly_units":
                continue

            row[f"forecast_{k}"] = v

        # ====================================================
        # HOURLY WEATHER VARIABLES
        # ====================================================

        hourly = forecast.get("hourly", {})

        for k, v in hourly.items():
            row[k] = first_or_none(v)

        rows.append(row)

# ============================================================
# EXPORT
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df):,} rows")
print(f"Saved CSV -> {OUTPUT_CSV}")
print(f"Columns -> {len(df.columns)}")

print("\nColumns:")
for c in df.columns:
    print("-", c)

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================


output_path = "/content/processed_merged.csv"

df1 = pd.read_csv(output_path, low_memory=False)

print(f"File1 rows: {len(df1):,}")

In [ ]:
import pandas as pd

df = pd.read_csv("processed_merged.csv")

print(len(df))          # number of rows
# or
print(df.shape[0])      # also number of rows

In [ ]:
import pandas as pd

# =========================================================
# LOAD DATA
# =========================================================

file_path = "/content/all_59k_accidents_complete_final_2.csv"
output_path = "/content/all_59k_accidents_complete_final_2_cleaned.csv"

df = pd.read_csv(file_path, low_memory=False)

print(f"Initial shape: {df.shape}")

# =========================================================
# DROP TRULY CONSTANT COLUMNS
# =========================================================

cols_to_drop = [
    "casualtyclass",
    "mode_of_travel"
]

# (optional safety check: show what exists)
actually_dropping = [c for c in cols_to_drop if c in df.columns]

print("\nDropping columns:")
print(actually_dropping)

df = df.drop(columns=actually_dropping, errors="ignore")

# =========================================================
# SAVE OUTPUT
# =========================================================

df.to_csv(output_path, index=False)

print(f"\nFinal shape: {df.shape}")
print(f"Saved cleaned file to: {output_path}")

In [ ]:
# Getting archive - 1 day before

#!/usr/bin/env python3

import csv
import json
import os
import time
from datetime import datetime, timezone, timedelta
import pandas as pd
import requests
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "all_rows_00_to_06.csv"
OUTPUT_JSONL = "open_meteo_archive_missing.jsonl"
ERRORS_CSV = "open_meteo_errors.csv"

# RAW_RESP_DIR = "responses_raw"
# ARCHIVE_RAW_DIR = os.path.join(RAW_RESP_DIR, "archive")

# os.makedirs(ARCHIVE_RAW_DIR, exist_ok=True)

REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
BACKOFF_FACTOR = 2.0
SLEEP_BETWEEN_REQUESTS = 0.0

# Hard stop for the run, so you stay under your monthly cap.
MONTHLY_CALL_BUDGET = int(os.getenv("OPEN_METEO_CALL_BUDGET", "2000000"))

# Set these via env if needed.
OPEN_METEO_API_KEY = os.getenv("OPEN_METEO_API_KEY", "").strip()
ARCHIVE_ENDPOINT = os.getenv(
    "OPEN_METEO_ARCHIVE_ENDPOINT",
    "https://archive-api.open-meteo.com/v1/archive",
)


# Open-Meteo controls
TIMEZONE = "UTC"
TIMEFORMAT = "iso8601"
CELL_SELECTION = "land"

# Use the provider’s combined default unless you want to override it.
ARCHIVE_MODELS = os.getenv("OPEN_METEO_ARCHIVE_MODELS", "best_match")

# If set, these override Open-Meteo’s default downscaling behavior.
# Leave blank to omit the parameter.
ELEVATION_OVERRIDE = os.getenv("OPEN_METEO_ELEVATION_OVERRIDE", "").strip()

# Batch size per request. Keeping it at 10 is conservative.
BATCH_SIZE = 10

session = requests.Session()
calls_made = 0


# ============================================================
# VARIABLE SETS
# ============================================================

# Archive hourly variables (verified from the current archive docs)
ARCHIVE_HOURLY_VARS = [
    "temperature_2m",
    "relativehumidity_2m",
    "dewpoint_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weathercode",
    "pressure_msl",
    "surface_pressure",
    "cloudcover",
    "cloudcover_low",
    "cloudcover_mid",
    "cloudcover_high",
    "windspeed_10m",
    "winddirection_10m",
    "windgusts_10m",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
    "windspeed_100m",
    "winddirection_100m",
    "soil_temperature_0_to_7cm",
    "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm",
    "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm",
    "soil_moisture_100_to_255cm",
]

ARCHIVE_DAILY_VARS = [
    "weathercode",
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_max",
    "apparent_temperature_min",
    "sunrise",
    "sunset",
    "daylight_duration",
    "sunshine_duration",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "precipitation_hours",
    "windspeed_10m_max",
    "windgusts_10m_max",
    "winddirection_10m_dominant",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

PRESSURE_LEVELS = [
    1000, 975, 950, 925, 900, 850, 800, 700, 600, 500,
    400, 300, 250, 200, 150, 100, 70, 50, 30,
]
PRESSURE_LEVEL_FAMILIES = [
    "temperature",
    "relative_humidity",
    "dew_point",
    "cloudcover",
    "wind_speed",
    "wind_direction",
    "geopotential_height",
]

# CSV columns
OUTPUT_COLUMNS = [
    "_row_index",
    "accident_datetime_utc",
    "source_latitude",
    "source_longitude",
    "archive_status",
    "archive_error",
    "archive_request_count",
    "archive_json",
]


# ============================================================
# HELPERS
# ============================================================

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def parse_accident_datetime_utc(row):
    raw_date = str(row.get("accidentdate", "")).strip()
    raw_time = str(row.get("time", "")).strip()

    if not raw_date:
        raise ValueError("missing accidentdate")
    if not raw_time:
        raise ValueError("missing time")

    try:
        date_part = datetime.strptime(raw_date, "%d-%b-%y")
    except Exception as exc:
        raise ValueError(f"invalid accidentdate format: {raw_date}") from exc

    try:
        time_part = datetime.strptime(raw_time, "%H:%M")
    except Exception as exc:
        raise ValueError(f"invalid time format: {raw_time}") from exc

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,
    )




def get_base_params(lat, lon, extra_params=None):
    params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "timezone": TIMEZONE,
        "timeformat": TIMEFORMAT,
        "cell_selection": CELL_SELECTION,
        "models": ARCHIVE_MODELS,
    }

    if ELEVATION_OVERRIDE != "":
        params["elevation"] = ELEVATION_OVERRIDE

    if OPEN_METEO_API_KEY:
        params["apikey"] = OPEN_METEO_API_KEY

    if extra_params:
        for k, v in extra_params.items():
            if k != "_endpoint" and v is not None:
                params[k] = v

    return params


def merge_payloads(target, source):
    if not target:
        target = {}

    for key, value in source.items():
        if key in {"hourly", "hourly_units", "daily", "daily_units", "current", "current_units", "minutely_15", "minutely_15_units"}:
            if key not in target:
                target[key] = {}
            if isinstance(value, dict):
                target[key].update(value)
        else:
            if key not in target:
                target[key] = value
    return target


def request_json(endpoint, params):
    global calls_made

    attempt = 0
    wait = 2.0
    last_url = None

    while attempt < MAX_RETRIES:
        try:
            calls_made += 1
            resp = session.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
            last_url = resp.url

            if resp.status_code == 429:
                pause = 60 if attempt < 2 else 300
                print(f"\n[!] Rate limited (429). Sleeping {pause}s before retry {attempt + 1}...")
                time.sleep(pause)
                attempt += 1
                continue

            if resp.status_code == 400:
                return False, {
                    "error": f"400 bad request: {resp.text}",
                    "url": resp.url
                }, resp.url

            resp.raise_for_status()
            return True, resp.json(), resp.url

        except Exception as exc:
            print(f"\n[!] Request error: {exc}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, {"error": "Max retries exceeded", "url": last_url}, last_url


def fetch_batched_section(endpoint, section_key, variables, base_params, batch_prefix):
    merged = {}
    urls = []
    request_count = 0

    for batch_idx, batch in enumerate(chunked(variables, BATCH_SIZE), start=1):
        params = dict(base_params)
        params[section_key] = ",".join(batch)

        ok, payload_or_err, url = request_json(endpoint, params)
        request_count += 1

        if not ok:
            return (
                False,
                merged,
                urls,
                request_count,
                payload_or_err.get("error", "request failed"),
            )

        payload = payload_or_err
        merged = merge_payloads(merged, payload)
        urls.append(url)

    return True, merged, urls, request_count, ""


def fetch_archive_for_row(row_idx, lat, lon, accident_dt):
    base_params = get_base_params(lat, lon, {"_endpoint": "archive"})
    archive_start_date = (accident_dt.date() - timedelta(days=1)).isoformat()
    archive_end_date = accident_dt.date().isoformat()

    base_params.update({
        "start_date": archive_start_date,
        "end_date": archive_end_date,
        "hourly": None,
        "daily": None,
    })

    archive = {}
    total_requests = 0
    errors = []

    ok_h, hourly_payload, hourly_urls, reqs, err = fetch_batched_section(
        ARCHIVE_ENDPOINT,
        "hourly",
        ARCHIVE_HOURLY_VARS,
        base_params,
        "hourly",
    )
    total_requests += reqs
    if ok_h:
        archive = merge_payloads(archive, hourly_payload)
    else:
        errors.append(f"hourly: {err}")

    ok_d, daily_payload, daily_urls, reqs, err = fetch_batched_section(
        ARCHIVE_ENDPOINT,
        "daily",
        ARCHIVE_DAILY_VARS,
        base_params,
        "daily",
    )
    total_requests += reqs
    if ok_d:
        archive = merge_payloads(archive, daily_payload)
    else:
        errors.append(f"daily: {err}")

    archive_meta = {
        "endpoint": ARCHIVE_ENDPOINT,
        "urls": hourly_urls + daily_urls,
        "requests": total_requests,
        "errors": errors,
    }
    return len(errors) == 0, archive_meta, archive



def empty_payload():
    return {}


# ============================================================
# MAIN LOOP
# ============================================================

def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    completed_rows = set()

    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    completed_rows.add(json.loads(line)["_row_index"])
                except Exception:
                    pass
    if not os.path.exists(OUTPUT_JSONL):
        open(OUTPUT_JSONL, "a").close()
    if os.path.exists(ERRORS_CSV):
        os.remove(ERRORS_CSV)

    errors_header_written = False

    success_rows = 0
    error_rows = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching accident rows"):
        if idx in completed_rows:
            continue
        if calls_made >= MONTHLY_CALL_BUDGET:
            print(
                f"\n[!] Monthly call budget reached "
                f"({calls_made} >= {MONTHLY_CALL_BUDGET}). Stopping."
            )
            break

        try:
            row = row.fillna("")
            lat_raw = row.get("latitude", row.get("source_latitude"))
            lon_raw = row.get("longitude", row.get("source_longitude"))

            if pd.isna(lat_raw) or pd.isna(lon_raw) or str(lat_raw).strip() == "" or str(lon_raw).strip() == "":
                raise ValueError("missing lat/lon")

            lat = float(str(lat_raw).strip())
            lon = float(str(lon_raw).strip())
            accident_dt = parse_accident_datetime_utc(row)

            archive_ok, archive_meta, archive_payload = fetch_archive_for_row(
                idx, lat, lon, accident_dt
            )


            out_row = {
                "_row_index": idx,
                "accident_datetime_utc": accident_dt.isoformat(),
                "source_latitude": lat,
                "source_longitude": lon,

                "archive_status": "ok" if archive_ok else "partial",
                "archive_error": archive_meta["errors"],
                "archive_request_count": archive_meta["requests"],
                "archive": archive_payload,
            }

            with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
                f.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            success_rows += 1

            if not archive_ok:
                with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as ef:
                    writer = csv.writer(ef)
                    if not errors_header_written:
                        writer.writerow(["row_index", "type", "message"])
                        errors_header_written = True
                    writer.writerow([
                        idx,
                        "partial",
                        f"archive={archive_meta['errors']}",
                    ])

        except Exception as exc:
            error_rows += 1
            with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as ef:
                writer = csv.writer(ef)
                if not errors_header_written:
                    writer.writerow(["row_index", "type", "message"])
                    errors_header_written = True
                writer.writerow([idx, "fatal", str(exc)])

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(
        f"\nDone. Rows saved: {success_rows}, errors: {error_rows}, "
        f"HTTP calls made: {calls_made}"
    )


if __name__ == "__main__":
    if not os.path.exists(INPUT_CSV):
        print(f"Error: {INPUT_CSV} not found in current directory.")
        raise SystemExit(1)

    run_fetcher()

In [ ]:
#!/usr/bin/env python3

import json
import os
import pandas as pd

INPUT_JSONL = "open_meteo_archive_missing.jsonl"
OUTPUT_CSV = "open_meteo_archive_missing.csv"

# ---------------------------------------------------------------------
# OUTPUT SCHEMA
# - hourly lags are exact offsets from the matched hour
# - last_3h / last_6h windows INCLUDE the matched hour
# - hourly match is the nearest hourly timestamp in the payload
# ---------------------------------------------------------------------

PROVENANCE_COLUMNS = [
    "_row_index",
    "accident_datetime_utc",
    "source_latitude",
    "source_longitude",
    "archive_status",
    "archive_error",
    "archive_request_count",
]

HOURLY_OPS = [
    ("temperature_2m", "avg"),
    ("relativehumidity_2m", "avg"),
    ("dewpoint_2m", "avg"),
    ("apparent_temperature", "avg"),
    ("precipitation", "sum"),
    ("rain", "sum"),
    ("snowfall", "sum"),
    ("snow_depth", "sum"),
    ("weathercode", "avg"),
    ("pressure_msl", "avg"),
    ("surface_pressure", "avg"),
    ("cloudcover", "avg"),
    ("cloudcover_low", "avg"),
    ("cloudcover_mid", "avg"),
    ("cloudcover_high", "avg"),
    ("windspeed_10m", "avg"),
    ("winddirection_10m", "avg"),
    ("windgusts_10m", "avg"),
    ("et0_fao_evapotranspiration", "avg"),
    ("vapour_pressure_deficit", "avg"),
    ("windspeed_100m", "avg"),
    ("winddirection_100m", "avg"),
    ("soil_temperature_0_to_7cm", "avg"),
    ("soil_temperature_7_to_28cm", "avg"),
    ("soil_temperature_28_to_100cm", "avg"),
    ("soil_temperature_100_to_255cm", "avg"),
    ("soil_moisture_0_to_7cm", "avg"),
    ("soil_moisture_7_to_28cm", "avg"),
    ("soil_moisture_28_to_100cm", "avg"),
    ("soil_moisture_100_to_255cm", "avg"),
]

DAILY_VARS = [
    "weathercode",
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_max",
    "apparent_temperature_min",
    "sunrise",
    "sunset",
    "daylight_duration",
    "sunshine_duration",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "precipitation_hours",
    "windspeed_10m_max",
    "windgusts_10m_max",
    "winddirection_10m_dominant",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

FEATURE_COLUMNS = [
    "archive_retrieved_latitude",
    "archive_retrieved_longitude",
    "archive_hour_match_gap_seconds",
]

for var, op in HOURLY_OPS:
    FEATURE_COLUMNS.extend([
        f"archive.hourly.{var}",
        f"archive.hourly.{var}_lag_1h",
        f"archive.hourly.{var}_lag_3h",
    ])
    if op == "avg":
        FEATURE_COLUMNS.extend([
            f"archive.hourly.{var}_avg_last_3h",
            f"archive.hourly.{var}_avg_last_6h",
        ])
    else:
        FEATURE_COLUMNS.extend([
            f"archive.hourly.{var}_sum_last_3h",
            f"archive.hourly.{var}_sum_last_6h",
        ])

for var in DAILY_VARS:
    FEATURE_COLUMNS.append(f"archive.daily.{var}")

OUTPUT_COLUMNS = PROVENANCE_COLUMNS + FEATURE_COLUMNS


def to_ts(x):
    ts = pd.to_datetime(x, utc=True, errors="coerce")
    return None if pd.isna(ts) else ts


def to_num(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return None


def mean_or_none(values):
    vals = [to_num(v) for v in values]
    vals = [v for v in vals if v is not None]
    return None if not vals else sum(vals) / len(vals)


def sum_or_none(values):
    vals = [to_num(v) for v in values]
    vals = [v for v in vals if v is not None]
    return None if not vals else sum(vals)


def build_hourly_index(hourly):
    raw_times = hourly.get("time", [])
    times = [to_ts(t) for t in raw_times]
    times = [t for t in times if t is not None]
    time_to_idx = {t.isoformat(): i for i, t in enumerate(times)}
    return times, time_to_idx


def nearest_hour_index(times, accident_ts):
    if not times or accident_ts is None:
        return None, None

    deltas = [abs(t - accident_ts) for t in times]
    idx = min(range(len(deltas)), key=lambda i: deltas[i])
    gap_seconds = int(deltas[idx].total_seconds())
    return idx, gap_seconds


def exact_hour_index(time_to_idx, ts):
    if ts is None:
        return None
    return time_to_idx.get(ts.isoformat())


def series_value(values, idx):
    if idx is None:
        return None
    if idx < 0 or idx >= len(values):
        return None
    return values[idx]


def window_values(hourly, time_to_idx, center_ts, var, hours_back):
    """
    Window includes center_ts and the prior hours_back-1 hours.
    For example:
      hours_back=3 -> [t-2h, t-1h, t]
      hours_back=6 -> [t-5h, t-4h, t-3h, t-2h, t-1h, t]
    """
    vals = []
    values = hourly.get(var, [])

    for offset in range(hours_back):
        ts = center_ts - pd.Timedelta(hours=offset)
        idx = exact_hour_index(time_to_idx, ts)
        v = series_value(values, idx)
        if v is not None and not pd.isna(v):
            vals.append(v)

    return vals


def build_row(obj):
    row = {k: obj.get(k) for k in PROVENANCE_COLUMNS}

    accident_ts = to_ts(obj.get("accident_datetime_utc"))
    archive = obj.get("archive") or {}
    hourly = archive.get("hourly") or {}
    daily = archive.get("daily") or {}

    row["archive_retrieved_latitude"] = archive.get("latitude")
    row["archive_retrieved_longitude"] = archive.get("longitude")

    times, time_to_idx = build_hourly_index(hourly)
    nearest_idx, gap_seconds = nearest_hour_index(times, accident_ts)
    row["archive_hour_match_gap_seconds"] = gap_seconds

    matched_hour_ts = times[nearest_idx] if nearest_idx is not None else None

    # Hourly features
    for var, op in HOURLY_OPS:
        values = hourly.get(var, [])
        matched_value = series_value(values, nearest_idx)

        row[f"archive.hourly.{var}"] = matched_value

        lag_1h_ts = matched_hour_ts - pd.Timedelta(hours=1) if matched_hour_ts is not None else None
        lag_3h_ts = matched_hour_ts - pd.Timedelta(hours=3) if matched_hour_ts is not None else None

        row[f"archive.hourly.{var}_lag_1h"] = series_value(values, exact_hour_index(time_to_idx, lag_1h_ts))
        row[f"archive.hourly.{var}_lag_3h"] = series_value(values, exact_hour_index(time_to_idx, lag_3h_ts))

        if matched_hour_ts is not None:
            window_3h = window_values(hourly, time_to_idx, matched_hour_ts, var, 3)
            window_6h = window_values(hourly, time_to_idx, matched_hour_ts, var, 6)
        else:
            window_3h = []
            window_6h = []

        if op == "avg":
            row[f"archive.hourly.{var}_avg_last_3h"] = mean_or_none(window_3h)
            row[f"archive.hourly.{var}_avg_last_6h"] = mean_or_none(window_6h)
        else:
            row[f"archive.hourly.{var}_sum_last_3h"] = sum_or_none(window_3h)
            row[f"archive.hourly.{var}_sum_last_6h"] = sum_or_none(window_6h)

    # Daily features: exact date match on the daily time series
    daily_times = [to_ts(t) for t in daily.get("time", [])]
    daily_times = [t for t in daily_times if t is not None]
    daily_index = {t.date().isoformat(): i for i, t in enumerate(daily_times)}

    accident_date = accident_ts.date().isoformat() if accident_ts is not None else None
    daily_match_idx = daily_index.get(accident_date)

    for var in DAILY_VARS:
        values = daily.get(var, [])
        row[f"archive.daily.{var}"] = series_value(values, daily_match_idx)

    return row


def main():
    rows = []

    with open(INPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            rows.append(build_row(obj))

    df = pd.DataFrame(rows)

    # Enforce column order and keep any unexpected extras at the end if they exist
    for col in OUTPUT_COLUMNS:
        if col not in df.columns:
            df[col] = None

    extras = [c for c in df.columns if c not in OUTPUT_COLUMNS]
    df = df[OUTPUT_COLUMNS + extras]

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved {len(df):,} rows -> {OUTPUT_CSV}")
    print(f"Columns: {len(df.columns)}")


if __name__ == "__main__":
    if not os.path.exists(INPUT_JSONL):
        raise SystemExit(f"Missing input file: {INPUT_JSONL}")
    main()

In [ ]:
import pandas as pd

# Input and output file paths
input_file = "open_meteo_archive_missing.csv"
output_file = "open_meteo_archive_missing_clean.csv"

# Columns to drop
cols_to_drop = ["archive_status", "archive_error", "archive_request_count"]

# Load CSV
df = pd.read_csv(input_file)

# Drop columns (ignore errors in case some don't exist)
df = df.drop(columns=cols_to_drop, errors="ignore")

# Save result
df.to_csv(output_file, index=False)

print("Done. Cleaned CSV saved to:", output_file)

In [ ]:
import logging
import pandas as pd

# =========================
# FILE PATHS
# =========================
file1_path = "/content/all_59k_accidents_complete_final_2_cleaned.csv"
file2_path = "/content/open_meteo_archive_missing_clean.csv"

FILE1_LAT = "latitude"
FILE1_LON = "longitude"
FILE1_DT  = "accident_time_utc"

FILE2_LAT = "source_latitude"
FILE2_LON = "source_longitude"
FILE2_DT  = "accident_datetime_utc"

# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("merge")


# =========================
# LOAD FILES
# =========================
logger.info("Loading files...")

df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

logger.info(f"file1 rows={len(df1)}")
logger.info(f"file2 rows={len(df2)}")


# =========================
# STRICT NORMALIZATION (EXACT MATCH MODE)
# =========================
def norm_latlon(s):
    return (
        pd.to_numeric(s, errors="coerce")
        .map(lambda x: format(x, ".10f") if pd.notnull(x) else None)
    )

def norm_dt(s):
    return (
        pd.to_datetime(s, errors="coerce", utc=True)
        .dt.strftime("%Y-%m-%dT%H:%M:%S+00:00")
    )


df1["_k_lat"] = norm_latlon(df1[FILE1_LAT])
df1["_k_lon"] = norm_latlon(df1[FILE1_LON])
df1["_k_dt"]  = norm_dt(df1[FILE1_DT])

df2["_k_lat"] = norm_latlon(df2[FILE2_LAT])
df2["_k_lon"] = norm_latlon(df2[FILE2_LON])
df2["_k_dt"]  = norm_dt(df2[FILE2_DT])


# =========================
# MERGE
# =========================
logger.info("Merging...")

merged = df1.merge(
    df2,
    on=["_k_lat", "_k_lon", "_k_dt"],
    how="left",
    indicator=True
)

matched = merged["_merge"].eq("both")

logger.info(f"Matched rows: {matched.sum()}")
logger.info(f"Unmatched rows: {(~matched).sum()}")


# =========================
# PRINT 4 UNMATCHED ROWS
# =========================
unmatched = merged.loc[~matched]

print("\n==============================")
print("4 UNMATCHED ROWS (DEBUG VIEW)")
print("==============================\n")

print(
    unmatched[
        [FILE1_LAT, FILE1_LON, FILE1_DT,
         "_k_lat", "_k_lon", "_k_dt", "_merge"]
    ].head(4)
)


# =========================
# EXTRA DEBUG (OPTIONAL BUT USEFUL)
# =========================
print("\n==============================")
print("CHECK: NULL KEYS")
print("==============================")

print("NaN lat file1:", df1["_k_lat"].isna().sum())
print("NaN lon file1:", df1["_k_lon"].isna().sum())
print("NaN dt  file1:", df1["_k_dt"].isna().sum())

print("NaN lat file2:", df2["_k_lat"].isna().sum())
print("NaN lon file2:", df2["_k_lon"].isna().sum())
print("NaN dt  file2:", df2["_k_dt"].isna().sum())

In [ ]:
import pandas as pd

# In Colab, replace this with your file path.
csv_path = "/content/all_59k_accidents_complete_final_2_cleaned.csv"
output_path = "/content/cleaned_file.csv"

# Load the CSV
df = pd.read_csv(csv_path)

# Find columns whose names contain "archive.daily"
dropped_columns = [col for col in df.columns if "archive.daily" in col]

# Drop them
df_cleaned = df.drop(columns=dropped_columns)

# Save the cleaned file
df_cleaned.to_csv(output_path, index=False)

# Summary
print(f"Dropped {len(dropped_columns)} columns:")
for col in dropped_columns:
    print(f"- {col}")

print(f"\nSaved cleaned CSV to: {output_path}")
print(f"Remaining columns: {df_cleaned.shape[1]}")